In [ ]:
# Data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Model saving
import joblib

# Suppress warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv("sample_1000.csv")

# Fix vessel name casing  (MEGAStar → Megastar, FINLANDIA → Finlandia)
df['ship'] = df['ship'].str.strip().str.title()

df.head()

In [ ]:
df.columns

In [ ]:
# IMPORTANT: Three different date formats exist in this dataset
# atd          → ISO 8601  (YYYY-MM-DD HH:MM:SS)
# etdSchedule  → DD/MM/YYYY
# ata          → MM/DD/YYYY  ← different! must use dayfirst=False

df['atd_dt']         = pd.to_datetime(df['atd'],         errors='coerce')
df['etdSchedule_dt'] = pd.to_datetime(df['etdSchedule'], dayfirst=True,  errors='coerce')
df['ata_dt']         = pd.to_datetime(df['ata'],         dayfirst=False, errors='coerce')
df['etaSchedule_dt'] = pd.to_datetime(df['etaSchedule'], dayfirst=True,  errors='coerce')

# Drop rows where departure time is missing
df = df.dropna(subset=['atd_dt', 'etdSchedule_dt']).reset_index(drop=True)

# Compute delays in minutes
df['dep_delay_min'] = (df['atd_dt'] - df['etdSchedule_dt']).dt.total_seconds() / 60
df['arr_delay_min'] = (df['ata_dt'] - df['etaSchedule_dt']).dt.total_seconds() / 60

# Remove outliers using IQR fence
q1, q3 = df['dep_delay_min'].quantile(0.25), df['dep_delay_min'].quantile(0.75)
iqr = q3 - q1
df.loc[(df['dep_delay_min'] < q1 - 3*iqr) | (df['dep_delay_min'] > q3 + 3*iqr), 'dep_delay_min'] = np.nan

# Exclude anchor pings (SOG < 1 knot = ship sitting at berth)
df['sog_valid'] = df['sog'].where(df['sog'] >= 1)

df.head()

In [ ]:
# Each voyage can have multiple GPS pings — group into one row per trip
df['voyage_key'] = (
    df['ship'].astype(str) + '_' +
    df['depPort'].astype(str) + '_' +
    df['arrPort'].astype(str) + '_' +
    df['etdSchedule_dt'].astype(str)
)

agg = df.groupby('voyage_key', sort=False).agg(
    ship          = ('ship',           'first'),
    depPort       = ('depPort',        'first'),
    arrPort       = ('arrPort',        'first'),
    etd_sched     = ('etdSchedule_dt', 'first'),
    eta_sched     = ('etaSchedule_dt', 'first'),
    ping_count    = ('sog',            'count'),
    mean_sog      = ('sog_valid',      'mean'),
    dep_delay_min = ('dep_delay_min',  'first'),
    arr_delay_min = ('arr_delay_min',  'last'),
).reset_index()

agg = agg.sort_values('etd_sched').reset_index(drop=True)

print(f"Raw pings  : {len(df):,}")
print(f"Voyages    : {len(agg):,}")

In [ ]:
# A voyage is DELAYED if it departed more than 5 minutes late
# 5 min threshold gives a realistic 10.4% delay rate to learn from

agg['is_delayed'] = (agg['dep_delay_min'] > 5).astype(int)

print("Delay mapping:")
print("  0 = On Time  (departed within 5 minutes of schedule)")
print("  1 = Delayed  (departed more than 5 minutes late)")
print()
print(agg['is_delayed'].value_counts())

In [ ]:
# Time features
agg['hour_of_dep']  = agg['etd_sched'].dt.hour
agg['day_of_week']  = agg['etd_sched'].dt.dayofweek
agg['month']        = agg['etd_sched'].dt.month
agg['is_weekend']   = (agg['day_of_week'] >= 5).astype(int)
agg['is_peak_hour'] = agg['hour_of_dep'].isin(range(7, 11)).astype(int)

# Cyclical encoding (so hour 23 and hour 0 are treated as close, not far apart)
agg['hour_sin'] = np.sin(2 * np.pi * agg['hour_of_dep'] / 24)
agg['hour_cos'] = np.cos(2 * np.pi * agg['hour_of_dep'] / 24)
agg['dow_sin']  = np.sin(2 * np.pi * agg['day_of_week'] / 7)
agg['dow_cos']  = np.cos(2 * np.pi * agg['day_of_week'] / 7)

# Port traffic (how many vessels departing same hour)
agg['hour_bucket']  = agg['etd_sched'].dt.floor('h')
traffic_map         = agg.groupby('hour_bucket')['ship'].count()
agg['port_traffic'] = agg['hour_bucket'].map(traffic_map)
agg['traffic_3h']   = agg['port_traffic'].rolling(3, min_periods=1).mean()
agg['traffic_6h']   = agg['port_traffic'].rolling(6, min_periods=1).mean()
agg['prev_traffic'] = agg['port_traffic'].shift(1).fillna(agg['port_traffic'].mean())

# Vessel history lag features (shift(1) = previous voyage, no data leakage)
agg = agg.sort_values(['ship', 'etd_sched']).reset_index(drop=True)
agg['prev_dep_delay']   = agg.groupby('ship')['dep_delay_min'].shift(1).fillna(0)
agg['rolling_delay_3v'] = (
    agg.groupby('ship')['dep_delay_min']
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
).fillna(0)
agg['prev_was_late']     = (agg['prev_dep_delay'] > 5).astype(int)
agg['is_slow_crossing']  = (agg['mean_sog'].fillna(0) < 15).astype(int)
agg['is_hel_to_tal']     = agg['depPort'].str.contains('FIH', case=False).astype(int)
agg['sched_duration_min']= (
    (agg['eta_sched'] - agg['etd_sched']).dt.total_seconds() / 60
).clip(lower=60)
agg['vessel_avg_delay']  = agg.groupby('ship')['dep_delay_min'].transform('mean')
agg['mean_sog']          = agg.groupby('ship')['mean_sog'].transform(
    lambda x: x.fillna(x.median())).fillna(agg['mean_sog'].median())

agg = agg.sort_values('etd_sched').reset_index(drop=True)
agg.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

MODEL_FEATURES = [
    'mean_sog', 'is_slow_crossing',
    'hour_of_dep', 'day_of_week', 'month', 'is_weekend', 'is_peak_hour',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'port_traffic', 'traffic_3h', 'traffic_6h', 'prev_traffic',
    'vessel_avg_delay', 'prev_dep_delay', 'rolling_delay_3v', 'prev_was_late',
    'is_hel_to_tal', 'sched_duration_min'
]
TARGET = 'is_delayed'

model_df = agg[MODEL_FEATURES + [TARGET, 'ship', 'etd_sched', 'dep_delay_min']].dropna(subset=[TARGET]).copy()

# Chronological split — never shuffle time-series data
split_idx = int(len(model_df) * 0.70)
X = model_df[MODEL_FEATURES]
y = model_df[TARGET].astype(int)
X_train, X_test = X.iloc[:split_idx].copy(), X.iloc[split_idx:].copy()
y_train, y_test = y.iloc[:split_idx].copy(), y.iloc[split_idx:].copy()

# Leak-free vessel_avg_delay: compute only on train rows
train_vessel_delay = model_df.iloc[:split_idx].groupby('ship')['dep_delay_min'].mean()
fallback = float(train_vessel_delay.mean())
X_train['vessel_avg_delay'] = model_df.iloc[:split_idx]['ship'].map(train_vessel_delay).fillna(fallback).values
X_test['vessel_avg_delay']  = model_df.iloc[split_idx:]['ship'].map(train_vessel_delay).fillna(fallback).values

model = RandomForestClassifier(
    n_estimators=300, max_depth=6, min_samples_leaf=8,
    class_weight='balanced', random_state=42, n_jobs=-1
)
model.fit(X_train, y_train)

preds = model.predict(X_test)

print(classification_report(y_test, preds, target_names=['On Time', 'Delayed']))

In [ ]:
import matplotlib.pyplot as plt

importances = model.feature_importances_

# Sort for cleaner chart
sorted_idx  = importances.argsort()
sorted_feats = [MODEL_FEATURES[i] for i in sorted_idx]
sorted_imps  = importances[sorted_idx]

colors = ['#ef4444' if v > 0.10 else '#0099bb' if v > 0.05 else '#94a3b8'
          for v in sorted_imps]

plt.figure(figsize=(10, 8))
bars = plt.barh(sorted_feats, sorted_imps, color=colors, edgecolor='white')
plt.title("Feature Importance — What Drives Delay Predictions?", fontsize=13, fontweight='bold')
plt.xlabel("Importance Score")

# Label the top bars
for bar, val in zip(bars, sorted_imps):
    if val > 0.05:
        plt.text(val + 0.002, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score, RocCurveDisplay

probs = model.predict_proba(X_test)[:, 1]

# Confusion Matrix
cm = confusion_matrix(y_test, preds)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['On Time', 'Delayed'],
    yticklabels=['On Time', 'Delayed']
)
plt.title("Confusion Matrix", fontsize=13, fontweight='bold')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

# ROC Curve
auc = roc_auc_score(y_test, probs)
print("ROC-AUC Score:", round(auc, 4))

RocCurveDisplay.from_estimator(model, X_test, y_test)
plt.title(f"ROC Curve  (AUC = {auc:.3f})", fontsize=13, fontweight='bold')
plt.show()

In [ ]:
print("Training set shape :", X_train.shape)
print("Test set shape     :", X_test.shape)
print("Target shape       :", y.shape)
print()
print("Delay rate (train) :", f"{y_train.mean():.1%}")
print("Delay rate (test)  :", f"{y_test.mean():.1%}")

In [ ]:
import joblib

model_bundle = {
    'model':          model,
    'features':       MODEL_FEATURES,
    'threshold':      0.30,
    'vessel_delays':  train_vessel_delay.to_dict(),
    'fallback_delay': fallback,
}

joblib.dump(model_bundle, "model.pkl")
print("Model saved → model.pkl ✅")

In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score, RocCurveDisplay
import matplotlib.pyplot as plt

# Delay distribution histogram
clean = agg['dep_delay_min'].dropna().clip(-30, 30)

plt.figure(figsize=(8, 4))
plt.hist(clean, bins=40, color='#0099bb', edgecolor='white', alpha=0.85)
plt.axvline(0,  color='#ef4444', linewidth=2, linestyle='--', label='On Time Line')
plt.axvline(5,  color='#f59e0b', linewidth=2, linestyle='--', label='Delay Threshold (5 min)')
plt.title("Distribution of Departure Delay", fontsize=13, fontweight='bold')
plt.xlabel("Delay (minutes)")
plt.ylabel("Number of Voyages")
plt.legend()
plt.tight_layout()
plt.show()

# ROC Curve
probs = model.predict_proba(X_test)[:, 1]
auc   = roc_auc_score(y_test, probs)
print("ROC-AUC Score:", round(auc, 4))

RocCurveDisplay.from_estimator(model, X_test, y_test)
plt.title(f"ROC Curve  (AUC = {auc:.3f})", fontsize=13, fontweight='bold')
plt.show()

In [ ]:
# Encode vessel as integer ID
vessel_map = {v: i for i, v in enumerate(sorted(agg['ship'].unique()))}
agg['vessel_id'] = agg['ship'].map(vessel_map)

# Encode route direction
agg['route'] = agg['depPort'] + ' → ' + agg['arrPort']

print("Vessel encoding:")
for v, i in vessel_map.items():
    print(f"  {i} = {v}")

print()
print("Routes in dataset:")
print(agg['route'].value_counts())

In [ ]:
X = model_df[MODEL_FEATURES]

print("Final feature matrix shape:", X.shape)
print()
print("Features used:")
for i, f in enumerate(MODEL_FEATURES, 1):
    print(f"  {i:2}. {f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

num_cols = ['dep_delay_min', 'mean_sog', 'port_traffic']
titles   = ['Departure Delay (minutes)', 'Crossing Speed (knots)', 'Port Traffic (vessels/hour)']

for col, title in zip(num_cols, titles):
    plt.figure(figsize=(7, 4))
    sns.histplot(agg[col].dropna(), kde=True, color='#0099bb')
    plt.title(f"Distribution of {title}", fontsize=12, fontweight='bold')
    plt.xlabel(title)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

In [ ]:
for col, title in zip(num_cols, titles):
    plt.figure(figsize=(7, 3))
    sns.boxplot(x=agg[col].dropna(), color='#0099bb')
    plt.title(f"Boxplot of {title}", fontsize=12, fontweight='bold')
    plt.xlabel(title)
    plt.tight_layout()
    plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
vessel_counts = agg['ship'].value_counts()
sns.barplot(x=vessel_counts.index, y=vessel_counts.values,
            palette=['#ef4444','#f59e0b','#10b981','#0099bb'])
plt.title("Voyage Count by Vessel", fontsize=12, fontweight='bold')
plt.xlabel("Vessel")
plt.ylabel("Number of Voyages")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
route_counts = agg['route'].value_counts()
sns.barplot(x=route_counts.index, y=route_counts.values,
            palette=['#0099bb','#f59e0b'])
plt.title("Voyage Count by Route Direction", fontsize=12, fontweight='bold')
plt.xlabel("Route")
plt.ylabel("Number of Voyages")
plt.tight_layout()
plt.show()

In [ ]:
X.isnull().sum()

In [ ]:
before = len(X)
X = X.dropna()
print(f"Rows before : {before}")
print(f"Rows after  : {len(X)}")
print(f"Dropped     : {before - len(X)}")

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='mean')
X[num_cols] = imputer.fit_transform(X[num_cols])

print("Missing values after imputation:")
print(X[num_cols].isnull().sum())

In [ ]:
# Confirm all binary features are correctly encoded as 0/1
binary_cols = ['is_weekend', 'is_peak_hour', 'is_slow_crossing',
               'prev_was_late', 'is_hel_to_tal', 'is_delayed']

print("Binary feature value counts:")
for col in binary_cols:
    if col in agg.columns:
        print(f"  {col}: {dict(agg[col].value_counts().sort_index())}")

In [ ]:
plt.figure(figsize=(9, 7))
corr_cols = ['dep_delay_min', 'mean_sog', 'port_traffic',
             'hour_of_dep', 'is_weekend', 'is_peak_hour', 'is_delayed']
sns.heatmap(
    agg[corr_cols].corr(),
    annot=True, fmt='.2f', cmap='coolwarm', center=0,
    linewidths=0.3
)
plt.title("Correlation Matrix", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

y = model_df['is_delayed'].astype(int)

# Chronological split — 70% train, 30% test, no shuffling
split_idx = int(len(model_df) * 0.70)

X_train = model_df[MODEL_FEATURES].iloc[:split_idx].copy()
X_test  = model_df[MODEL_FEATURES].iloc[split_idx:].copy()
y_train = y.iloc[:split_idx]
y_test  = y.iloc[split_idx:]

print(f"Train: {len(X_train)} rows  |  Test: {len(X_test)} rows")
print(f"Train delay rate: {y_train.mean():.1%}")
print(f"Test  delay rate: {y_test.mean():.1%}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=8,
    class_weight='balanced',
    random_state=42
)
model.fit(X_train, y_train)
print("Model training complete ✅")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

preds = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds, target_names=['On Time', 'Delayed']))

In [ ]:
from sklearn.metrics import roc_auc_score

probs = model.predict_proba(X_test)[:, 1]
print("ROC-AUC:", roc_auc_score(y_test, probs))

In [ ]:
print(X_train.columns)

In [ ]:
print(model.feature_names_in_)

In [ ]:
print(agg['ship'].unique())
print(agg['route'].unique())

In [ ]:
# Congestion Heatmap — Avg Departure Delay by Vessel × Hour of Day
heatmap_data = (
    agg.groupby(['ship', 'hour_of_dep'])['dep_delay_min']
    .mean()
    .unstack('hour_of_dep')
    .reindex(columns=range(24))
    .fillna(0)
    .round(1)
)

plt.figure(figsize=(16, 4))
sns.heatmap(
    heatmap_data,
    cmap='RdYlGn_r',
    center=0,
    annot=True,
    fmt='.0f',
    linewidths=0.3,
    cbar_kws={'label': 'Avg Delay (min)'}
)
plt.title("Congestion Heatmap — Avg Departure Delay (min) by Vessel × Hour",
          fontsize=13, fontweight='bold')
plt.xlabel("Hour of Day (0–23)")
plt.ylabel("")
plt.tight_layout()
plt.show()